# Limburg map to Heerlen edge table csv

In [6]:
# Build edge-level connectivity + travel time tables from the routing graph
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox

In [ ]:
# Build a drivable graph for Heerlen + 1 km in every direction
heerlen_boundary = ox.geocode_to_gdf("Heerlen, Limburg, Netherlands")

# Buffer in a metric CRS so 1 km is accurate
heerlen_boundary_m = heerlen_boundary.to_crs(epsg=3857)
expanded_geom_m = heerlen_boundary_m.geometry.iloc[0].buffer(250)
expanded_geom = gpd.GeoSeries([expanded_geom_m], crs="EPSG:3857").to_crs(epsg=4326).iloc[0]

route_graph = ox.graph_from_polygon(expanded_geom, network_type="drive")
print("Loaded graph for Heerlen + 250 m buffer.")

In [ ]:
# 1) Add speed and travel time attributes (travel_time in seconds)
route_graph_tt = ox.add_edge_speeds(route_graph)
route_graph_tt = ox.add_edge_travel_times(route_graph_tt)

# 2) Convert graph edges to a table
edges_gdf = ox.graph_to_gdfs(route_graph_tt, nodes=False, edges=True).reset_index()

# Keep common attributes if present
cols = ["u", "v", "key", "name", "highway", "length", "speed_kph", "travel_time", "geometry"]
edge_table = edges_gdf[[c for c in cols if c in edges_gdf.columns]].copy()

# Clean name/highway fields (they can be lists)
def _to_text(value):
    if isinstance(value, list):
        return " | ".join(map(str, value))
    return value

if "name" in edge_table.columns:
    edge_table["name"] = edge_table["name"].apply(_to_text)
if "highway" in edge_table.columns:
    edge_table["highway"] = edge_table["highway"].apply(_to_text)

# Add human-readable travel time
if "travel_time" in edge_table.columns:
    edge_table["travel_time_min"] = edge_table["travel_time"] / 60.0

print(f"Directed road segments (edges): {len(edge_table):,}")
edge_table.head(10)

# Export expanded-area edge table
edge_table.to_csv("../output/hearlen_edge_table_2.csv", index=False)
print("Saved: ../output/hearlen_edge_table_2.csv")

Directed road segments (edges): 7,183
Saved: ../output/hearlen_edge_table_2.csv
